# 🎥 MoTSR — real-life video upscaling

**Made by [xyether](https://github.com/XYETHER) 🤍**

A 2× temporal model that uses nearby frames and its previous results. The spatial backbone is **[MoSRv2 by umzi2](https://github.com/umzi2/MoSRV2)**.

1. Choose a **T4 GPU** in Runtime → Change runtime type.
2. Run **Setup**, then **Upscale** and upload a video.
3. Download your result ✨

> 🧪 Still in development: results can include artifacts and vary by clip.

[Examples, source code & weights](https://github.com/XYETHER/MoTSR) · MIT license


In [ ]:
#@title #1. **✨ Setup Environment**
#@markdown Select **Runtime → Change runtime type → T4 GPU**, then press ▶️.
#@markdown This downloads MoTSR's ready-to-use engines. You only need to run setup once per session.
import sys, subprocess, importlib, hashlib, urllib.request, time
from pathlib import Path
import torch

if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):
    raise RuntimeError("Please select a T4 GPU in Runtime → Change runtime type, then rerun this cell.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "tensorrt==11.0.0.114", "opencv-python-headless", "tqdm"])
if not __import__("shutil").which("ffmpeg"):
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-yqq", "ffmpeg"])

MOTSR_ROOT = Path("/content/MoTSR")
MOTSR_MODELS = MOTSR_ROOT / "models"
MOTSR_MODELS.mkdir(parents=True, exist_ok=True)
HASHES = {'MoTSR_2x_seed_T4.engine': 'e8af6ee5c9f7b2cce8dbee71efdb3fc5d4700cb6129cae83b5897a938e788e84', 'MoTSR_2x_recurrent_T4.engine': '43b13065ec01a94e2397f255e44bb46e78002429bd9d12d81393a124555df74b', 'motsr_video.py': 'a7d25addffb89ef00a802c673d2389ca6e511fe9618bb497096207ed2319aa87'}
RELEASE = "https://github.com/XYETHER/MoTSR/releases/download/v1.0.0/"

def fetch_verified(url, destination, sha256):
    if destination.exists() and hashlib.sha256(destination.read_bytes()).hexdigest() == sha256:
        return
    temporary = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=120) as response, temporary.open("wb") as target:
                __import__("shutil").copyfileobj(response, target)
            if hashlib.sha256(temporary.read_bytes()).hexdigest() != sha256:
                raise RuntimeError("Download checksum mismatch. Please rerun setup.")
            temporary.replace(destination)
            return
        except Exception:
            temporary.unlink(missing_ok=True)
            if attempt == 2:
                raise
            time.sleep(3)

for name, checksum in HASHES.items():
    destination = MOTSR_ROOT / name if name.endswith(".py") else MOTSR_MODELS / name
    fetch_verified(RELEASE + name, destination, checksum)
if str(MOTSR_ROOT) not in sys.path:
    sys.path.insert(0, str(MOTSR_ROOT))
importlib.invalidate_caches()
import motsr_video
importlib.reload(motsr_video)
print("✅ MoTSR is ready! Upload a video in the next cell.")


In [ ]:
#@title #2. **🚀 Upscale (Real-life video)**
#@markdown Leave **video_path** empty to upload your video, or paste its path in this runtime.
video_path = "" #@param {type:"string"}
#@markdown **Speed_Boost** keeps the model input within 720p. Turn it off for input up to 1080p.
Speed_Boost = True #@param {type:"boolean"}
#@markdown **Force_1080p** resizes the finished output to fit 1920×1080.
Force_1080p = False #@param {type:"boolean"}
codec = "hevc_nvenc" #@param ["h264_nvenc", "hevc_nvenc"]
#@markdown CRF defaults to **6** and ranges from **0 to 10**. Lower values mean less compression and larger files. Use **h264_nvenc** for wider playback support.
crf_value = 6 #@param {type:"slider", min:0, max:10, step:1}
auto_download = True #@param {type:"boolean"}

from google.colab import files
from pathlib import Path
import gc, torch

if "MOTSR_MODELS" not in globals():
    raise RuntimeError("Please run the setup cell first.")
path = Path(video_path.strip()) if video_path.strip() else None
if path is None:
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No video was uploaded.")
    if len(uploaded) != 1:
        raise ValueError("Please upload one video at a time.")
    path = Path(next(iter(uploaded)))
    del uploaded
if not path.is_file():
    raise FileNotFoundError(f"Video not found: {path}")
output = MOTSR_ROOT / "results" / (path.stem + "_MoTSR_2x.mp4")
try:
    report = motsr_video.upscale_video(path, output, MOTSR_MODELS,
        speed_boost=Speed_Boost, force_1080p=Force_1080p, codec=codec, quality=crf_value)
finally:
    gc.collect()
    torch.cuda.empty_cache()
print(f"✨ Finished! Your video is here: {output}")
if auto_download:
    files.download(str(output))


### 💡 A few tips

- The output is 2× the model input. Speed Boost may reduce larger inputs first.
- Scene changes reset the recurrent memory automatically; this is a simple heuristic.
- No interpolation is added. Original frame timing is preserved; audio is retained as AAC when present.
- Processing uses temporary PNG frames. Longer videos need free runtime disk space.
- These engines target T4 + TensorRT 11.0.0.114. Other GPUs can use the PyTorch source from the repo.

Notebook adapted from [Xyether Anime Upscaler](https://github.com/XYETHER/Xyether-Anime-Upscaler).
